# Lesson 2 : Trace Agent

In actual development, you will likely need to perform detailed tracing and correct the context - such as, customizing instructions, setting-up appropriate tools, adding memories, etc.  
Agent Framework integrates with tracing and logging capabilities by using OpenTelemetry standard, and you can then configure to work with a wide variety of logging platforms - such as, Aspire Dashboard, Jaeger, Prometheus, etc. (You can also output trace and logs in your console.)

In this exercise, we'll trace a agent built in Lesson 1, and send logs to Azure Application Insights through Microsoft Foundry.

## 1. Create and connect to an Application Insights resource

In order for tracing, you should prepare an Application Insights resource in Microsoft Foundry.  
Before you start tracing, run the following steps.

1. Open [Azure Portal](https://portal.azure.com) and create a new Application Insights resource.
2. Open Foundry Portal (new portal), click "Operate" menu, and select "Admin" in left-side navigation.
3. Select your project, and connect to above Application Insights resource by clicking "Add connection".

## 2. Tracing your agent

Same as Lesson 1, we initialize a client object for the agent.

In [1]:
from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv()

credential = AzureCliCredential()
client = FoundryChatClient(credential=credential)

Also same as Lesson 1, now we create an agent with local function tools as follows.

In [2]:
from typing import Annotated
from pydantic import Field
from random import randint
from agent_framework import Agent, tool

# define local tools
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

# connect to the agent
agent = Agent(
    name="BasicWeatherAgent",
    client=client,
    instructions="You are an agent about weather information.",
    tools=[get_weather, get_temperature])

Run agent with trace enabled as follows.

For the tracing with Application Insights, you can simply use built-in ```configure_azure_monitor()``` in client. (This method internally detects the connected Application Insights resource, invokes ```configure_azure_monitor()``` in Azure observability SDK, and invokes built-in ```enable_instrumentation()``` in Agent Framework SDK.)

> Note : For other tracing, you can use built-in ```configure_otel_providers()``` which reads related environments and configures. (You can also manually configure exporters, providers, and instrumentation for OpenTelemetry tracing.)  
> See [here](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/observability) for examples.

In [3]:
from IPython.display import Markdown, display
from agent_framework.observability import get_tracer
from opentelemetry.trace import SpanKind
from opentelemetry.trace.span import format_trace_id

await client.configure_azure_monitor(
    enable_live_metrics=False,
    enable_sensitive_data=True,
)

with get_tracer().start_as_current_span("Weather Agent Test") as current_span:
    print(f"Trace ID: {format_trace_id(current_span.get_span_context().trace_id)}")

    result = await agent.run("Tell me the weather and temperature in Osaka.")
    display(Markdown(result.text))

Trace ID: 35f2b2aa88729fbc965edd1b86a19ee1


Osaka: **Cloudy**, **11 °C**.

You can then view the collected trace in Application Insights or Azure Monitor. (For the first time, it may take a while...)  
For example, let's see and check logs calling OpenAI Responses API invoked from Agent Framework, as follows.

1. Go to Application Insights resource in [Azure Portal](https://portal.zure.com), and select your resource.
2. Expand "Investigate" in left-side navigator and click "Agents".
3. Click item in "Tool calls" or "Models" section.
4. Select a target trace session.
5. Expand call navigation in left-side, and select one of "LLM" items or "GenAI" items. (See below picture.)

![Trace view in Application Insights](./assets/appinsight_trace.png)

> Note : To view the input text and the generated output text, set ```enable_sensitive_data=True``` in ```configure_azure_monitor()```.